# Tests Notebook
Notebook ini berisi pengujian minimal 5 test case untuk helper Dijkstra dan
tiga algoritma utama (CDSSSD, MDMSMD, EAMDSP).

In [ ]:
import unittest

from algorithms import (
    PathNotFoundError,
    dijkstra_shortest_path,
    run_cdsssd,
    run_mdmsmd,
    run_eamdsp,
)


In [ ]:
graph = {
    'A': [('B', 4), ('C', 2)],
    'B': [('A', 4), ('C', 1), ('D', 5), ('E', 12)],
    'C': [('A', 2), ('B', 1), ('D', 8), ('E', 10)],
    'D': [('B', 5), ('C', 8), ('E', 2), ('F', 6)],
    'E': [('B', 12), ('C', 10), ('D', 2), ('F', 2)],
    'F': [('D', 6), ('E', 2)],
}

disconnected_graph = {
    'A': [('B', 1)],
    'B': [('A', 1)],
    'C': [('D', 1)],
    'D': [('C', 1)],
}


In [ ]:
class TestShortestPathAlgorithms(unittest.TestCase):
    """Unit tests untuk validasi hasil dan edge cases."""

    def test_01_dijkstra_basic(self):
        result = dijkstra_shortest_path(graph, 'A', 'E')
        self.assertEqual(result['path'], ['A', 'C', 'B', 'D', 'E'])
        self.assertEqual(result['cost'], 10.0)

    def test_02_cdsssd_duplicate_destination_kept(self):
        result = run_cdsssd(graph, 'A', ['C', 'C'])
        self.assertEqual(len(result['destination_results']), 2)
        self.assertEqual(result['visit_order'], ['C', 'C'])

    def test_03_mdmsmd_source_inside_destinations(self):
        result = run_mdmsmd(graph, 'A', ['A', 'C'])
        self.assertEqual(result['segments'][0]['cost'], 0.0)
        self.assertEqual(result['segments'][0]['path'], ['A'])

    def test_04_eamdsp_returns_route_and_segments(self):
        result = run_eamdsp(graph, 'A', ['F', 'C'])
        self.assertEqual(result['algorithm'], 'EAMDSP')
        self.assertEqual(len(result['segments']), 2)
        self.assertEqual(result['visit_order'][0], 'C')

    def test_05_unreachable_raises_exception(self):
        with self.assertRaises(PathNotFoundError):
            run_mdmsmd(disconnected_graph, 'A', ['D'])

    def test_06_negative_weight_rejected(self):
        bad_graph = {
            'A': [('B', -1)],
            'B': [('A', -1)],
        }
        with self.assertRaises(ValueError):
            dijkstra_shortest_path(bad_graph, 'A', 'B')

    def test_07_path_merge_no_duplicate_join(self):
        result = run_mdmsmd(graph, 'A', ['C', 'B'])
        # join C->B tidak boleh menduplikasi C pada full_path
        self.assertEqual(result['full_path'], ['A', 'C', 'B'])


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestShortestPathAlgorithms)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


## Ringkasan Hasil Test
Jika semua test berstatus `ok`, maka implementasi utama sudah memenuhi
kebutuhan fungsional dasar dan edge case penting.